In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    Trainer, 
    TrainingArguments
)


import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

import proj_consts_
# from proj_consts_ import *

# easy conversion between pitch and int token, and finger and int token
# print("pitch_to_int_mapping: ", finger_mappings.pitch_to_int_mapping)
# print("int_to_pitch_mapping: ", finger_mappings.int_to_pitch_mapping)

# print("finger_to_int_mapping: ", finger_mappings.finger_to_int_mapping)
# print("int_to_finger_mapping: ", finger_mappings.int_to_finger_mapping)

pitch_to_int_mapping = proj_consts_.pitch_to_int_mapping
int_to_pitch_mapping = proj_consts_.int_to_pitch_mapping
finger_to_int_mapping = proj_consts_.finger_to_int_mapping
int_to_finger_mapping = proj_consts_.int_to_finger_mapping

In [2]:
def convert_features_to_text(x_list):
    """
    Example conversion of numeric features -> a textual sequence.
    Adjust this to your preference.
    
    x_list: list of [pitch_int, onset_time, offset_time, onset_velocity, offset_velocity, channel]
    Returns a single string like: "pitch 60 onset 0.0 offset 1.5 velocity 90 channel 0 ..." for each token
    or a simpler condensed version. 
    """
    tokens = []
    for row in x_list:
        pitch_int    = row[0]
        onset_time   = row[1]
        offset_time  = row[2]
        onset_vel    = row[3]
        offset_vel   = row[4]
        channel      = row[5]
        
        # Create a compact textual representation
        token = f"<p{pitch_int}> <on{int(onset_time)}> <off{int(offset_time)}> <vel{int(onset_vel)}> <c{int(channel)}>"
        tokens.append(token)
    
    # Join with spaces or use another delimiter
    return " ".join(tokens)

def convert_fingers_to_text(y_list):
    """
    Convert integer finger labels to a textual sequence
    e.g. [1,2,3,4] -> "1 2 3 4"
    """
    return " ".join(str(f) for f in y_list)

In [4]:
class T5FingeringDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_length=512):
        """
        sequences: list of (X_list, Y_list)
        tokenizer: T5Tokenizer
        max_length: max token length for T5
        """
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        X_list, y_list = self.sequences[idx]
        
        # Convert numeric arrays to text
        input_text  = convert_features_to_text(X_list)
        target_text = convert_fingers_to_text(y_list)

        # Tokenize (we only return the raw strings here; the collator can do the padding)
        item = {
            "input_text":  input_text,
            "target_text": target_text
        }
        return item


def t5_data_collator(batch, tokenizer, max_length=512):
    """
    batch: list of dictionary items from our Dataset
    tokenizer: T5Tokenizer
    """
    input_texts  = [item["input_text"] for item in batch]
    target_texts = [item["target_text"] for item in batch]

    # Tokenize input and target sequences
    model_inputs = tokenizer(
        input_texts,
        max_length=max_length,
        truncation=True,
        padding="longest",
        return_tensors="pt"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            target_texts,
            max_length=max_length,
            truncation=True,
            padding="longest",
            return_tensors="pt"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

class T5DataCollator:
    def __init__(self, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __call__(self, batch):
        return t5_data_collator(batch, self.tokenizer, self.max_length)


# PRETRAINING

In [5]:
import os
import pandas as pd

def load_encoded_sequences(folder_path, pitch_to_int_mapping, finger_to_int_mapping):
    """
    Loads all fingering files from a given folder_path,
    encodes them into (X_list, y_list) pairs, and returns a list of these pairs.
    """
    # Collect DataFrames
    all_dataframes = []
    
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        # e.g. "001-01_fingering.txt" => "001-01", you might need a safer split if filenames differ
        fingering_label, _ = filename.split('_')  # "001-01"
        
        # If you need piece_id, fingering_type separately:
        # piece_id, fingering_type = fingering_label.split('-')
        
        # Check if this is a valid file
        if os.path.isfile(file_path):
            df = pd.read_table(
                file_path, 
                sep="\t", 
                skiprows=1, 
                names=[
                    "noteID", "onset_time", "offset_time", "spelled_pitch",
                    "onset_velocity", "offset_velocity", "channel", "finger_number"
                ]
            )
            all_dataframes.append(df)
    
    print(f"Found {len(all_dataframes)} fingering files in {folder_path}")

    # Now encode each DataFrame into (X_list, y_list) pairs
    raw_encoded_sequences = []
    for df in all_dataframes:
        X_list = []
        y_list = []
        for row in df.itertuples(index=False):
            spelled_pitch = row.spelled_pitch
            finger_str    = str(row.finger_number)

            # Convert spelled pitch and finger to integer tokens
            pitch_int  = pitch_to_int_mapping.get(spelled_pitch, 0) 
            finger_int = finger_to_int_mapping.get(finger_str, 0)

            # Feature row: (pitch, onset_time, offset_time, onset_vel, offset_vel, channel)
            feature_row = [
                pitch_int,
                float(row.onset_time),
                float(row.offset_time),
                float(row.onset_velocity),
                float(row.offset_velocity),
                float(row.channel)
            ]
            X_list.append(feature_row)
            y_list.append(finger_int)

        raw_encoded_sequences.append((X_list, y_list))
    
    return raw_encoded_sequences


In [8]:
# Specify folder paths
PATH_TO_NOISY_SEQUENCES = './ThumbSet_v1.0/FingeringFiles'
PATH_TO_CLEAN_SEQUENCES = './PianoFingeringDataset_v1.2/FingeringFiles'  

# Then, call the helper function to get the sequences
noisy_sequences = load_encoded_sequences(
    PATH_TO_NOISY_SEQUENCES, 
    pitch_to_int_mapping, 
    finger_to_int_mapping
)

high_quality_sequences = load_encoded_sequences(
    PATH_TO_CLEAN_SEQUENCES, 
    pitch_to_int_mapping, 
    finger_to_int_mapping
)

print(f"Noisy sequences: {len(noisy_sequences)}")
print(f"High quality sequences: {len(high_quality_sequences)}")


KeyboardInterrupt: 

In [ ]:
# Example: 
# noisy_sequences  -> your 2,500 "noisy" pieces
# high_quality_sequences -> your 300+ "clean" pieces






from sklearn.model_selection import train_test_split

train_noisy, val_noisy = train_test_split(noisy_sequences, test_size=0.1, random_state=42)

tokenizer = T5Tokenizer.from_pretrained("t5-small")

train_noisy_dataset = T5FingeringDataset(train_noisy, tokenizer, max_length=256)
val_noisy_dataset   = T5FingeringDataset(val_noisy, tokenizer, max_length=256)

data_collator = T5DataCollator(tokenizer, max_length=256)

model = T5ForConditionalGeneration.from_pretrained("t5-small")

pretrain_args = TrainingArguments(
    output_dir="pretrain_t5_noisy",   # Where checkpoints are saved
    overwrite_output_dir=True,
    num_train_epochs=5,               # Adjust as needed
    per_device_train_batch_size=2,    # Adjust for GPU memory
    per_device_eval_batch_size=2,
    evaluation_strategy="epoch",      # Evaluate at the end of each epoch
    save_strategy="epoch",            # Save checkpoint each epoch
    logging_steps=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),   # use FP16 if possible
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=pretrain_args,
    train_dataset=train_noisy_dataset,
    eval_dataset=val_noisy_dataset,
    data_collator=data_collator
)

# ---- 1) Pretrain on noisy dataset
trainer.train()

# Save your pretrained model
trainer.save_model("pretrained_t5_noisy")
